# WP4v2 — Notebook 3 : Tests d'inférence

Pipeline d'inférence complet :
```
Image → MAE (full ou masqué) → z_i (196 tokens, 1024 dim)
      → f_theta (point-wise) → z_i' (196 tokens, 1024 dim, espace CLIP)
      → MLP connector LLaVA  → z_i'' (196 tokens, 4096 dim, espace LLM)
      → LLM avec template Vicuna → description textuelle
```

Tests réalisés :
1. Vérification discriminabilité après projection f_theta
2. Sanity check : comparaison LLaVA natif (576 tokens) vs MAE projeté (196 tokens)
3. Visualisation patch-level avec masquage

## 1. Chargement des modèles

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import ViTMAEModel, ViTImageProcessor
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor, LlamaTokenizer
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

# f_theta
class ProjectionMLP(nn.Module):
    def __init__(self, dim=1024, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(dim), nn.Linear(dim, dim), nn.GELU(),
            nn.Dropout(dropout), nn.LayerNorm(dim), nn.Linear(dim, dim),
        )
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

f_theta = ProjectionMLP().to(DEVICE)
f_theta.load_state_dict(torch.load('wp4v2_projection_best.pt'))
f_theta.eval()

norm_stats = torch.load('wp4v2_norm_stats.pt')
z_mean = norm_stats['mean'].cpu()
z_std  = norm_stats['std'].cpu()

# MAE
mae_processor = ViTImageProcessor(
    size={'height': 224, 'width': 224},
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)
mae_encoder = ViTMAEModel.from_pretrained('./vit-mae-large').to(DEVICE)
mae_encoder.eval()

# LLaVA complet (vision tower + MLP connector + LLM)
llava_full = LlavaForConditionalGeneration.from_pretrained(
    './llava-1.5-7b-hf', torch_dtype=torch.float16
).to(DEVICE)
llava_full.eval()

llm          = llava_full.language_model
vision_tower = llava_full.vision_tower
mlp_conn     = llava_full.multi_modal_projector
clip_proc    = CLIPImageProcessor.from_pretrained('./llava-1.5-7b-hf')
tokenizer    = LlamaTokenizer.from_pretrained('./llava-1.5-7b-hf', use_fast=False)

print(f'VRAM : {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('Tous les modeles charges')


## 2. Fonctions utilitaires

In [ ]:
# Template Vicuna obligatoire pour LLaVA-1.5
SYSTEM    = ('A chat between a curious user and an artificial intelligence assistant. '
             'The assistant gives helpful, detailed, and polite answers to the user questions.')
USER_TEXT = 'Describe this image in one sentence.'
BEFORE    = f'{SYSTEM} USER: '
AFTER     = f'\n{USER_TEXT} ASSISTANT:'

before_ids    = tokenizer(BEFORE, return_tensors='pt', add_special_tokens=True).input_ids.to(DEVICE)
after_ids     = tokenizer(AFTER,  return_tensors='pt', add_special_tokens=False).input_ids.to(DEVICE)
before_embeds = llm.get_input_embeddings()(before_ids).half()
after_embeds  = llm.get_input_embeddings()(after_ids).half()


def project_tokens(tokens):
    """
    Applique f_theta point-wise sur une sequence de tokens MAE.
    tokens : Tensor (N, 1024) sur CPU ou GPU
    Retourne : Tensor (N, 1024) dans l'espace CLIP, L2-normalise
    """
    t_norm = (tokens.cpu().float() - z_mean) / z_std  # normalisation
    with torch.no_grad():
        projected = f_theta(t_norm.to(DEVICE))
    return projected.cpu().float()  # (N, 1024)


def llm_describe(visual_tokens_clip):
    """
    Pipeline complet : tokens CLIP (N, 1024) -> MLP connector -> LLM -> description.
    visual_tokens_clip : Tensor (N, 1024) dans l'espace CLIP
    """
    # MLP connector : (N, 1024) -> (N, 4096)
    with torch.no_grad():
        visual_llm = mlp_conn(visual_tokens_clip.half().to(DEVICE))  # (N, 4096)

    # Assemblage : [SYSTEM USER:] + [visual tokens] + [question ASSISTANT:]
    inputs_embeds = torch.cat([
        before_embeds,
        visual_llm.unsqueeze(0),
        after_embeds
    ], dim=1)  # (1, N+seq_len, 4096)

    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


def encode_mae_full(image_pil):
    """MAE full encoding — retourne 196 tokens (1024 dim) sur CPU."""
    inp = mae_processor(images=image_pil, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    with torch.no_grad():
        out = mae_encoder(**inp, noise=torch.zeros(1, 196).to(DEVICE))
    return out.last_hidden_state[0, 1:].cpu().float()  # (196, 1024)


def encode_mae_masked(image_pil, seed=42):
    """MAE masque — retourne 49 tokens visibles, le masque, et les positions."""
    inp = mae_processor(images=image_pil, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    gen = torch.Generator().manual_seed(seed)
    noise = torch.rand(1, 196, generator=gen).to(DEVICE)
    with torch.no_grad():
        out = mae_encoder(**inp, noise=noise)
    patch_tokens = out.last_hidden_state[0, 1:].cpu().float()  # (49, 1024)
    mask         = out.mask[0].cpu()
    visible_ids  = torch.where(mask == 0)[0].tolist()
    return patch_tokens, mask, visible_ids


def describe_llava_native(image_pil):
    """Pipeline LLaVA natif (576 tokens CLIP) — reference ground truth."""
    inp = clip_proc(images=image_pil, return_tensors='pt', do_rescale=True)
    pix = inp['pixel_values'].to(DEVICE).half()
    with torch.no_grad():
        vis     = vision_tower(pix).last_hidden_state[:, 1:]  # (1, 576, 1024)
        vis_llm = mlp_conn(vis)[0]                            # (576, 4096)
    inputs_embeds = torch.cat([before_embeds, vis_llm.unsqueeze(0), after_embeds], dim=1)
    with torch.no_grad():
        out_ids = llm.generate(
            inputs_embeds=inputs_embeds, max_new_tokens=50,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()


print('Fonctions definies')


## 3. Vérification discriminabilité après projection f_θ

In [ ]:
ds = load_dataset('parquet', data_files={
    'validation': './imagenet100/data/validation-*.parquet',
})

projections, mae_vecs = [], []
print(f'{"Classe":<40} {"cos sim MAE":>12} {"cos sim proj":>13}')
print('-' * 68)

for i in range(5):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB').resize((224, 224))
    tokens    = encode_mae_full(image_pil)              # (196, 1024)
    z_img     = tokens.mean(dim=0)                      # (1024,)
    z_proj    = project_tokens(z_img.unsqueeze(0))[0]   # (1024,)
    mae_vecs.append(F.normalize(z_img.unsqueeze(0), dim=-1)[0])
    projections.append(z_proj)
    print(f'{item["text"][:38]:<40}')

mae_mat  = torch.stack(mae_vecs)
proj_mat = torch.stack(projections)

print('\nSimilarites cosinus AVANT projection (espace MAE) :')
print((mae_mat @ mae_mat.T).numpy().round(4))
print('\nSimilarites cosinus APRES projection (espace CLIP) :')
print((proj_mat @ proj_mat.T).numpy().round(4))


## 4. Sanity check — LLaVA natif vs MAE projeté (196 tokens)

In [ ]:
N_IMAGES = 5
print(f'{"Classe":<38} {"LLaVA natif (576)":<45} {"MAE -> f_theta -> MLP (196)"}')
print('-' * 120)

for i in range(N_IMAGES):
    item      = ds['validation'][i]
    image_pil = item['image'].convert('RGB')
    label     = item['text'][:35]

    # LLaVA natif
    desc_llava = describe_llava_native(image_pil)

    # MAE -> f_theta -> MLP connector -> LLM
    image_224  = image_pil.resize((224, 224))
    mae_tokens = encode_mae_full(image_224)         # (196, 1024)
    clip_space = project_tokens(mae_tokens)         # (196, 1024) via f_theta
    desc_mae   = llm_describe(clip_space)           # MLP connector -> LLM

    print(f'{label:<38} {desc_llava:<45} {desc_mae}')


## 5. Visualisation patch-level avec masquage

In [ ]:
IMG_IDX = 0
SEED    = 42

item      = ds['validation'][IMG_IDX]
image_pil = item['image'].convert('RGB').resize((224, 224))
label_txt = item['text']

# Encodage masque
patch_tokens, mask, visible_ids = encode_mae_masked(image_pil, seed=SEED)
print(f'Classe : {label_txt} | Patches visibles : {len(visible_ids)}')

# Projeter les 49 tokens visibles via f_theta
clip_tokens = project_tokens(patch_tokens)  # (49, 1024)

# Description LLM depuis les tokens masques projectes
desc_masked = llm_describe(clip_tokens)
print(f'Description depuis 49 tokens masques : {desc_masked}')

# Description LLM depuis les 196 tokens full (pour comparaison)
image_pil_full = item['image'].convert('RGB').resize((224, 224))
full_tokens    = encode_mae_full(image_pil_full)   # (196, 1024)
clip_full      = project_tokens(full_tokens)        # (196, 1024)
desc_full      = llm_describe(clip_full)
print(f'Description depuis 196 tokens full  : {desc_full}')


In [ ]:
def extract_patch(image_pil, patch_id, patch_size=16, grid_size=14):
    row = patch_id // grid_size; col = patch_id % grid_size
    return image_pil.crop((col*patch_size, row*patch_size,
                           (col+1)*patch_size, (row+1)*patch_size))

# Visualisation : image masquee + grille des patches visibles
# avec description LLM depuis chaque patch individuel
N_SHOW = 10
step   = max(1, len(visible_ids) // N_SHOW)
subset_ids   = visible_ids[::step][:N_SHOW]
subset_tokens = clip_tokens[::step][:N_SHOW]  # tokens projectes correspondants

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW*2.5, 5))
fig.suptitle(f'Description LLM par patch — {label_txt}', fontsize=12)

for col, (patch_id, clip_tok) in enumerate(zip(subset_ids, subset_tokens)):
    crop = extract_patch(image_pil, patch_id).resize((64, 64), resample=0)
    axes[0, col].imshow(crop)
    axes[0, col].set_title(f'p{patch_id}', fontsize=7)
    axes[0, col].axis('off')

    # Description LLM depuis ce seul token
    desc = llm_describe(clip_tok.unsqueeze(0))
    axes[1, col].text(0.5, 0.5, desc[:60], ha='center', va='center',
                      fontsize=6, wrap=True, transform=axes[1, col].transAxes)
    axes[1, col].axis('off')
    print(f'p{patch_id:3d} : {desc}')

plt.tight_layout()
plt.savefig('wp4v2_patch_descriptions.png', dpi=150, bbox_inches='tight')
plt.show()
